### First, import the repo and any other library needed

Also install dependencies from the environment.yml in the repository

In [1]:
%pip install pandas numpy pillow tqdm torchvision ipywidgets
%pip install albumentations==1.4.11 albucore==0.0.16 efficientnet-pytorch==0.7.1 pytorchcv==0.0.67 opencv-python-headless==4.10.0.84

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 66.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 90.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 83.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 86.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 69.9 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 73.9 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 79.2 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 80.8 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 59.8 MB/s  0:00:01m0:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 89.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 78.5 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 90.8 MB/s  0:00:00
   ━━

In [2]:
!git clone https://github.com/polimi-ispl/synthetic-image-detection.git
## And also get into it with cd
%cd synthetic-image-detection

fatal: destination path 'synthetic-image-detection' already exists and is not an empty directory.
/workspace/data/AG/lozio_baresi/notebooks/synthetic-image-detection


In [1]:
## If repo is already there:
# !git pull
# %pip install -e .
%cd synthetic-image-detection

/workspace/data/AG/lozio_baresi/notebooks/synthetic-image-detection


In [2]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import torch

## For monkey patch if needed
import random
from utils.blazeface import BlazeFace, FaceExtractor
import albumentations as A

INFO:albumentations.check_version:A new version of Albumentations is available: 2.0.8 (you have 1.4.11). Upgrade using: pip install --upgrade albumentations


### Now get the weigths

In [3]:
!wget -O synth_vs_real.pth "https://www.dropbox.com/scl/fi/na1nff3ehp4kz98n1vtdj/synth_vs_real.pth?rlkey=3k4336bs5uqasr75jwz3666di&dl=1"

--2026-09-03 06:56:55--  https://www.dropbox.com/scl/fi/na1nff3ehp4kz98n1vtdj/synth_vs_real.pth?rlkey=3k4336bs5uqasr75jwz3666di&dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.69.18, 2620:100:6025:18::a27d:4512
Connecting to www.dropbox.com (www.dropbox.com)|162.125.69.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://uc8bf1c68ffc51a07594b3ce9fe4.dl-eu.dropboxusercontent.com/cd/0/inline/DHb4B79cx9QUZkPgrzW02FjkcJw-FBP_AcxvWqAIgNTnnxJ3o93xMERI3aN5gOzv5CqI64hvVlTxBxaMLkCW3FIl19F-78DxN0YP_AXMTzTg-Qe9Xs3ET-aCrlvrMg3k2Eggz5HrcIZPbU7pbWlduaP3/file?dl=1# [following]
--2026-09-03 06:56:56--  https://uc8bf1c68ffc51a07594b3ce9fe4.dl-eu.dropboxusercontent.com/cd/0/inline/DHb4B79cx9QUZkPgrzW02FjkcJw-FBP_AcxvWqAIgNTnnxJ3o93xMERI3aN5gOzv5CqI64hvVlTxBxaMLkCW3FIl19F-78DxN0YP_AXMTzTg-Qe9Xs3ET-aCrlvrMg3k2Eggz5HrcIZPbU7pbWlduaP3/file?dl=1
Resolving uc8bf1c68ffc51a07594b3ce9fe4.dl-eu.dropboxusercontent.com (uc8bf1c68ffc51a07594b3ce9fe4.dl-eu.dropboxuserc

### Consts and Paths here

The fodlers contianing the images

In [4]:
## PREVIOUS DATASET
VAL_DIRECTORY: str = "../../jpegai_dataset/validation/rec"
OG_DIRECTORY: str = "../../dataset/sub_validation"

OUTPUT_RESULT_CSV: str = "../real_vs_synth.csv"
LABEL_PATH: str = "../../jpegai_dataset/validation/validation.csv"
LABEL_CSV: Path = Path(LABEL_PATH)

## NEW DATASET
#NEW_JPEGAI_VAL: str = "../../jpegai_dataset/new_val"
#NEW_OG_VAL: str = "../../dataset/new_dataset/val"
#NEW_RESULT_CSV: str = "../NEW_real_vs_synth.csv"
NEW_FACE_RESULT_CSV: str = "../NEW_FACE_real_vs_synth.csv"

NEW_JPEGAI_VAL: str = "../../jpegai_dataset/extracted_10k/test"
NEW_OG_VAL: str = "../../dataset/extracted_10k/test"
NEW_RESULT_CSV: str = "../10k_real_vs_synth.csv"
#NEW_FACE_RESULT_CSV: str = "../10k_FACE_real_vs_synth.csv"

Let's confirm the paths are ok

In [5]:
to_test: list[str] = [NEW_JPEGAI_VAL, NEW_OG_VAL]

for p in to_test:
    if os.path.exists(p) and Path(p).is_dir():
        print("Path is valid!")
    else:
        print("Path does not exist: " + str(p))

Path is valid!
Path is valid!


In [6]:
## Also define the function to extract all images in a folder. it will return a dict with key 
## the name of any subfolder and as value the list of the images. If there a no subfolders, 
## the key is the name of the main fodler
def extract_images(main_folder: str) -> dict:
    if not main_folder:
        return {}
        
    main_path = Path(main_folder)
    img_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
    result = {}
    
    if not main_path.exists():
        print(f"Directory not found: {main_folder}")
        return result
        
    has_subfolders = any(p.is_dir() for p in main_path.iterdir())
    
    if not has_subfolders:
        images = [str(p) for p in main_path.iterdir() if p.suffix.lower() in img_exts]
        if images:
            result[main_path.name] = images
    else:
        for subfolder in main_path.iterdir():
            if subfolder.is_dir():
                images = [str(p) for p in subfolder.iterdir() if p.suffix.lower() in img_exts]
                if images:
                    result[subfolder.name] = images
                    
    return result

All qualities. JPEGAI append this at the of the name of the file

In [6]:
QUALITIES: list[str] = \
    [
        "012",
        "025",
        "050",
        "075",
        "100"
    ]

## Now execute it

Start by importing the detector

In [10]:
from test_real_vs_synthetic_singleimg import RealvsSyntheticDetector

In [11]:
# Helper to load image as array like the reference script expects
def load_image_as_array(path):
    img = Image.open(path)
    if img.mode not in ('RGB', 'L'):
        img = img.convert('RGB')
        
    return np.asarray(img)

The detector is not acceptable "as is" because it would instantiate the detector for every image, which is obviously not fine. So instead we iterate all images and get the score

### OLD

#### NO FACE TEST

In [ ]:
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
from pathlib import Path

## Instantiate detector. No need to "face only"
detector = RealvsSyntheticDetector(device='cuda:0', M=600, select_face_test=False)

## Extract original images from OG_DIRECTORY
og_images_dict = extract_images(OG_DIRECTORY)
val_dir_path = Path(VAL_DIRECTORY)
rows = []

## Now iterate all original images
for folder_name, images in og_images_dict.items():
    for img_path in tqdm(images, desc=f"Processing {folder_name}"):
        path_obj = Path(img_path)
        og_name = path_obj.stem
            
        row = {'Original_Image': path_obj.name, 'Folder': folder_name}
        
        # 1. Score for Original Image
        try:
            img_arr = load_image_as_array(img_path)
            score = float(detector.synth_img_detection(img=img_arr))
            row['Original_Score'] = score
        except Exception as e:
            row['Original_Score'] = None
            print(f"Error processing original {img_path}: {e}")
            
        # 2. Score for JPEGAI Compressed Images
        for quality in QUALITIES:
            jai_name = f"JAI_{og_name}_{quality}.png"
            
            # Find the JAI image in VAL_DIRECTORY
            jai_path_direct = val_dir_path / jai_name
            jai_path_sub = val_dir_path / path_obj.parent.name / jai_name
            
            if jai_path_direct.exists():
                jai_path = jai_path_direct
            elif jai_path_sub.exists():
                jai_path = jai_path_sub
            else:
                # Fallback: search recursively in VAL_DIRECTORY if not in obvious places
                matches = list(val_dir_path.rglob(jai_name))
                jai_path = matches[0] if matches else jai_path_direct
            
            if jai_path.exists():
                try:
                    img_arr = load_image_as_array(str(jai_path))
                    score = float(detector.synth_img_detection(img=img_arr))
                    row[f'Quality_{quality}_Score'] = score
                except Exception as e:
                    row[f'Quality_{quality}_Score'] = None
                    print(f"Error processing compressed {jai_path}: {e}")
            else:
                row[f'Quality_{quality}_Score'] = None
                
        rows.append(row)

# Save the scores to a CSV
### AI
if rows:
    df = pd.DataFrame(rows)
    
    # Merge with the ground truth labels if the CSV exists
    labels_csv_path = Path(LABEL_PATH)
    if labels_csv_path.exists():
        labels_df = pd.read_csv(labels_csv_path)
        
        # Create a temporary stem column for robust merging (ignores .jpg vs .png extension differences)
        df['temp_stem'] = df['Original_Image'].apply(lambda x: Path(x).stem)
        labels_df['temp_stem'] = labels_df['OG_IMG'].apply(lambda x: Path(x).stem)
        
        # Merge and clean up
        df = df.merge(labels_df, on='temp_stem', how='left')
        df = df.drop(columns=['temp_stem', 'OG_IMG'])
        
        print(f"Merged ground truth labels from {labels_csv_path}")
    else:
        print(f"Ground truth labels CSV not found at {labels_csv_path}. Proceeding without it.")

    df.to_csv(OUTPUT_RESULT_CSV, index=False)
    print(f"Saved scores to {OUTPUT_RESULT_CSV}.csv")
    display(df.head())
else:
    print("No images were processed. Please check OG_DIRECTORY and VAL_DIRECTORY.")

<All keys matched successfully>
Model for real vs synthetic image detection loaded!


Processing sub_validation:   0%|          | 0/200 [00:00<?, ?it/s]

#### FACE TEST

In [10]:
import pandas as pd
import numpy as np
import random
import torch
import albumentations as A
from PIL import Image
from tqdm.notebook import tqdm
from pathlib import Path

from test_real_vs_synthetic_singleimg import RealvsSyntheticDetector
from utils.blazeface import BlazeFace, FaceExtractor

# Helper to load image as array like the reference script expects
def load_image_as_array(path):
    img = Image.open(path)
    if img.mode not in ('RGB', 'L'):
        img = img.convert('RGB')
    return np.asarray(img)

# --- START OF MONKEY PATCH ---
# Global cache for the detector to avoid loading it 1200 times
_cached_face_detector = None
_cached_face_extractor = None

def patched_synth_img_detection(self, img: np.array) -> torch.Tensor:
    """
        Detection pipeline:
            1. 800 patches are randomly extracted from the image
            2. The patches are normalized
            3. All patches are processed through the real vs synthetic detector
            4. A majority voting on the scores obtained is performed to take a decision on the overall image,
            according to the parameter M which specifies how many patches to aggregate
            (for more information, see: https://arxiv.org/pdf/2407.10736)

        :param img: np.array, probe to analyze
        :return: torch.Tensor, synthetic image detection score:
        if score > 0, the image is detected as being synthetic
        if score < 0, the image is detected as being real
        """

    # set the seeds for the random extraction of patches
    random.seed(21)
    np.random.seed(21)
    torch.manual_seed(21)
    
    # Check on image format
    if img.ndim < 3:
        print('Gray scale image, converting to RGB')
        img2 = np.zeros((img.shape[0], img.shape[1], 3), dtype=np.uint8)
        img2[:, :, 0] = img
        img2[:, :, 1] = img
        img2[:, :, 2] = img
        img = img2.copy()
    if img.shape[2] > 3:
        print('Omitting alpha channel')
        img = img[:, :, :3]
    
    if self.select_face_test:
    
        ## -----> HERE, we prevent loading again  <------------
        global _cached_face_detector, _cached_face_extractor
        if _cached_face_detector is None:
            # Load face detector
            _cached_face_detector = BlazeFace()
            _cached_face_detector.load_weights("utils/blazeface/blazeface.pth")
            _cached_face_detector.load_anchors("utils/blazeface/anchors.npy")
    
        face_detector = _cached_face_detector

        if _cached_face_extractor is None:
            _cached_face_extractor = FaceExtractor(facedet=face_detector)
        
        face_extractor = _cached_face_extractor
    
        # Face detector is the same used in https://github.com/polimi-ispl/icpr2020dfdc
        # Check the linked repository for more information
    
        #  Pipeline for face extraction
        ############################################################################################################
    
        # Split the image into several tiles. Resize the tiles to 128x128.
        tiles, resize_info = face_extractor._tile_frames(frames=np.expand_dims(img, 0),
                                                         target_size=face_detector.input_size)
        # tiles has shape (num_tiles, target_size, target_size, 3)
        # resize_info is a list of four elements [resize_factor_y, resize_factor_x, 0, 0]
        # Run the face detector. The result is a list of PyTorch tensors,
        # one for each tile in the batch.
        detections = face_detector.predict_on_batch(tiles, apply_nms=False)
        # Convert the detections from 128x128 back to the original image size.
        image_size = (img.shape[1], img.shape[0])
        detections = face_extractor._resize_detections(detections, face_detector.input_size, resize_info)
        detections = face_extractor._untile_detections(1, image_size, detections)
        # The same face may have been detected in multiple tiles, so filter out overlapping detections.
        detections = face_detector.nms(detections)
    
        # Crop the faces out of the original frame.
        frameref_detections = face_extractor._add_margin_to_detections(detections[0], image_size, 0.5)
        faces = face_extractor._crop_faces(img, frameref_detections)
    
        # Add additional information about the frame and detections.
        scores = list(detections[0][:, 16])
        frame_dict = {"faces": faces,
                      "scores": scores,
                      }
        # consider at most the two best detected faces
        if len(faces) > 1:
            faces = [faces[x] for x in np.argsort(scores)]
            faces = [faces[-2], faces[-1]]
        # if only one face is detected, consider it
        elif len(faces) == 1:
            faces = [frame_dict['faces'][-1]]
        # if a face has not been detected, consider the entire img
        else:
            faces = [img]
    
        ############################################################################################################
    
        # define the list containing all the analyzed patches (for all the considered faces)
        all_patches = []
        for face in faces:
    
            # if the face size is smaller than 256 x 256, perform a little bit of upscaling to enlarge its size
            if face.shape[0] < 256 or face.shape[1] < 256:
                face = A.SmallestMaxSize(max_size=256, interpolation=1, always_apply=True, p=1)(image=face)['image']
    
            # randomly extract the patches:
            patches = [self.cropper(image=face)['image'] for x in range(800)]
            all_patches.extend(patches)
    
        # if the number of patches is too high (due to multiple faces detected), we still keep 800 patches
        if len(all_patches) > 800:
            # shuffle the patch-list
            random.shuffle(all_patches)
            all_patches = all_patches[:800]
    
    # if select_face_test = False
    else:
    
        # if the image size is smaller than 256 x 256, perform a little bit of upscaling to enlarge its size
        if img.shape[0] < 256 or img.shape[1] < 256:
            img = A.SmallestMaxSize(max_size=256, interpolation=1, always_apply=True, p=1)(image=img)['image']
    
        # extract patches from the image
        all_patches = [self.cropper(image=img)['image'] for x in range(800)]
    
    # Normalize the patches
    transf_patch_list = [self.trans(image=patch)['image'] for patch in all_patches]
    
    # Synthetic image detection
    ################################################################################################################
    
    # Compute scores
    transf_patch_tensor = torch.stack(transf_patch_list, dim=0).to(self.device)
    with torch.no_grad():
        patch_scores = self.net(transf_patch_tensor)
    
    # aggregate the scores to compute the final image score
    img_score = torch.mean(torch.sort(patch_scores[:, 1])[0][-self.M:])
    
    return img_score

# Apply monkey patch!
RealvsSyntheticDetector.synth_img_detection = patched_synth_img_detection
print("Successfully patched RealvsSyntheticDetector to cache BlazeFace.")
# --- END OF MONKEY PATCH ---

## Instantiate detector WITH face detection enabled
detector = RealvsSyntheticDetector(device='cuda:0', M=600, select_face_test=True)

## Extract original images from OG_DIRECTORY
og_images_dict = extract_images(OG_DIRECTORY)
val_dir_path = Path(VAL_DIRECTORY)
rows = []
i = 0

## Now iterate all original images
for folder_name, images in og_images_dict.items():
    for img_path in tqdm(images, desc=f"Processing {folder_name}"):
        path_obj = Path(img_path)
        og_name = path_obj.stem
            
        row = {'Original_Image': path_obj.name, 'Folder': folder_name}
        
        # 1. Score for Original Image
        try:
            img_arr = load_image_as_array(img_path)
            print(f"Processing: {i}. {img_path}")
            score = float(detector.synth_img_detection(img=img_arr))
            row['Original_Score'] = score
        except Exception as e:
            row['Original_Score'] = None
            print(f"Error processing original {img_path}: {e}")
            
        # 2. Score for JPEGAI Compressed Images
        for quality in QUALITIES:
            jai_name = f"JAI_{og_name}_{quality}.png"

            print(f"Looking for: {jai_name}")
            
            # Find the JAI image in VAL_DIRECTORY
            jai_path_direct = val_dir_path / jai_name
            jai_path_sub = val_dir_path / path_obj.parent.name / jai_name
            
            if jai_path_direct.exists():
                jai_path = jai_path_direct
            elif jai_path_sub.exists():
                jai_path = jai_path_sub
            else:
                # Fallback: search recursively in VAL_DIRECTORY if not in obvious places
                matches = list(val_dir_path.rglob(jai_name))
                jai_path = matches[0] if matches else jai_path_direct
            
            if jai_path.exists():
                try:
                    img_arr = load_image_as_array(str(jai_path))
                    print("Found it! Processing...")
                    score = float(detector.synth_img_detection(img=img_arr))
                    print("Done!")
                    row[f'Quality_{quality}_Score'] = score
                except Exception as e:
                    row[f'Quality_{quality}_Score'] = None
                    print(f"Error processing compressed {jai_path}: {e}")
            else:
                row[f'Quality_{quality}_Score'] = None
        i += 1
        rows.append(row)

# Save the scores to a CSV
if rows:
    df = pd.DataFrame(rows)
    
    # Merge with the ground truth labels if the CSV exists
    labels_csv_path = Path(LABEL_PATH)
    if labels_csv_path.exists():
        labels_df = pd.read_csv(labels_csv_path)
        
        # Create a temporary stem column for robust merging
        df['temp_stem'] = df['Original_Image'].apply(lambda x: Path(x).stem)
        labels_df['temp_stem'] = labels_df['OG_IMG'].apply(lambda x: Path(x).stem)
        
        # Merge and clean up
        df = df.merge(labels_df, on='temp_stem', how='left')
        df = df.drop(columns=['temp_stem', 'OG_IMG'])
        
        print(f"Merged ground truth labels from {labels_csv_path}")
    else:
        print(f"Ground truth labels CSV not found at {labels_csv_path}. Proceeding without it.")

    df.to_csv(OUTPUT_RESULT_CSV, index=False)
    print(f"Saved scores to {OUTPUT_RESULT_CSV}.csv")
    display(df.head())
else:
    print("No images were processed. Please check OG_DIRECTORY and VAL_DIRECTORY.")


Successfully patched RealvsSyntheticDetector to cache BlazeFace.
<All keys matched successfully>
Model for real vs synthetic image detection loaded!


Processing sub_validation:   0%|          | 0/200 [00:00<?, ?it/s]

Processing: 0. ../../dataset/sub_validation/f5b10a5455.png
Looking for: JAI_f5b10a5455_012.png
Found it! Processing...
Done!
Looking for: JAI_f5b10a5455_025.png
Found it! Processing...
Done!
Looking for: JAI_f5b10a5455_050.png
Found it! Processing...
Done!
Looking for: JAI_f5b10a5455_075.png
Found it! Processing...
Done!
Looking for: JAI_f5b10a5455_100.png
Found it! Processing...
Done!
Processing: 1. ../../dataset/sub_validation/569f079c51.png
Looking for: JAI_569f079c51_012.png
Found it! Processing...
Done!
Looking for: JAI_569f079c51_025.png
Found it! Processing...
Done!
Looking for: JAI_569f079c51_050.png
Found it! Processing...
Done!
Looking for: JAI_569f079c51_075.png
Found it! Processing...
Done!
Looking for: JAI_569f079c51_100.png
Found it! Processing...
Done!
Processing: 2. ../../dataset/sub_validation/164ae55a3c.png
Looking for: JAI_164ae55a3c_012.png
Found it! Processing...
Done!
Looking for: JAI_164ae55a3c_025.png
Found it! Processing...
Done!
Looking for: JAI_164ae55a3c_050

,Original_Image,Folder,Original_Score,Quality_012_Score,Quality_025_Score,Quality_050_Score,Quality_075_Score,Quality_100_Score,ID_IMG,IS_SYNTH
0,f5b10a5455.png,sub_validation,1.865147,3.350943,1.503896,1.551071,1.023170,2.077648,5152,0
1,569f079c51.png,sub_validation,-0.083564,2.907053,2.262594,1.560088,1.060010,0.149994,7113,1
2,164ae55a3c.png,sub_validation,1.253086,3.863961,2.772190,0.726932,0.891406,0.649668,2016,1
3,68bed27b2c.png,sub_validation,1.207237,3.922090,2.368495,0.858877,0.656169,1.561996,4392,1
4,aed0f2e1be.png,sub_validation,1.169931,2.469474,1.461266,2.094415,2.154569,1.508347,3009,1


#### Lastly, the accuracy

In [11]:
# Load the CSV if df is not already in memory, or just to be safe
df_results = pd.read_csv(OUTPUT_RESULT_CSV)

# Make sure the ground truth column actually exists before calculating
if 'IS_SYNTH' not in df_results.columns:
    print("Error: 'IS_SYNTH' ground truth column not found in the CSV. Did the merge succeed?")
else:
    # Drop rows where we don't have a ground truth label
    df_eval = df_results.dropna(subset=['IS_SYNTH']).copy()
    
    # 1 is Synthetic/Fake, 0 is Real
    ground_truth = df_eval['IS_SYNTH'].astype(int)
    
    # List of all score columns we want to evaluate
    score_columns = ['Original_Score'] + [f'Quality_{q}_Score' for q in QUALITIES]
    
    accuracies = {}
    
    for col in score_columns:
        if col in df_eval.columns:
            # Drop rows where the score is missing (e.g. if a quality image didn't exist)
            valid_rows = df_eval.dropna(subset=[col])
            
            if len(valid_rows) > 0:
                # Detector logic: Score > 0 means Synthetic (1), Score <= 0 means Real (0)
                predictions = (valid_rows[col] > 0).astype(int)
                actuals = valid_rows['IS_SYNTH'].astype(int)
                
                # Calculate accuracy
                accuracy = (predictions == actuals).mean()
                accuracies[col] = accuracy
            else:
                accuracies[col] = None
    
    # Display the final accuracies 
    acc_df = pd.DataFrame(list(accuracies.items()), columns=['Evaluation', 'Accuracy'])
    acc_df['Accuracy'] = acc_df['Accuracy'].apply(lambda x: f"{x:.2%}" if pd.notnull(x) else "N/A")
    print(acc_df)


          Evaluation Accuracy
0     Original_Score   54.00%
1  Quality_012_Score   68.50%
2  Quality_025_Score   66.50%
3  Quality_050_Score   64.00%
4  Quality_075_Score   61.00%
5  Quality_100_Score   60.00%


The results are quite unexpected. Let's see the number of reals and synth 

In [13]:
print(len(df_results[df_results['IS_SYNTH']==0]))
print(len(df_results[df_results['IS_SYNTH']==1]))

63
137


### New Dataset

#### NO FACE TEST

In [12]:
## Instantiate detector. No need to "face only"
detector = RealvsSyntheticDetector(device='cuda:0', M=600, select_face_test=False)

## Extract original images from OG_DIRECTORY
# og_images_dict = extract_images(OG_DIRECTORY)
val_dir_path = Path(VAL_DIRECTORY)
rows = []

REAL_FOLDER_LABEL: str = "real"
FAKE_FOLDER_LABEL: str = "fake"

def get_label(score: float) -> str:
    return FAKE_FOLDER_LABEL if score > 0 else REAL_FOLDER_LABEL

## Now iterate all original images
## To do so, we iterate the folder
for folder_name in [REAL_FOLDER_LABEL, FAKE_FOLDER_LABEL]:
    ## Take the path
    path = Path(NEW_OG_VAL) / folder_name

    ## JPGs are not actually allowed but is added as safety measure
    images = list(path.glob("*.png")) + list(path.glob("*.jpg"))
    
    ## Now iterate all imgs
    for img_path in tqdm(images):
        ## Create the new row
        row = {
            "Img_Name" : img_path.name,
            "Actual_Label" : folder_name
        }

        print(f"Processing: {Path(img_path).stem}")
        ## Now the score
        try:
            img_arr = load_image_as_array(img_path)
            score = float(detector.synth_img_detection(img=img_arr))
            row["Original_Score"] = score
            ## Also save the label directly
            row["Original_Label"] = get_label(score)
        except Exception as e:
            row["Original_Score"] = None
            row["Original_Label"] = ""
            print(f"Error processing original {img_path}: {e}")

        ## At this point, look for the JPEGAI version
        ## Remember to add "rec" to get the actual images
        jpegai_path = Path(NEW_JPEGAI_VAL) / folder_name / "rec"

        for qual in QUALITIES:
            ## First, get the name
            og_name = Path(img_path).stem
            jai_name = f"JAI_{og_name}_{qual}.png"
            print(f"Looking for: {jai_name}")
            jai_path: str = jpegai_path / jai_name
            jai_path: Path = Path(jai_path)

            ## We perfectly know where the img is now
            if jai_path.exists():
                try:
                    print("Found it! Processing...")
                    img_arr = load_image_as_array(str(jai_path))
                    score = float(detector.synth_img_detection(img=img_arr))
                    row[f"Quality_{qual}_Score"] = score
                    ## Also add the label with the same logic
                    row[f"Quality_{qual}_Label"] = get_label(score)
                except Exception as e:
                    row[f"Quality_{qual}_Score"] = None
                    row[f"Quality_{qual}_Label"] = ""
                    print(f"Error processing compressed {jai_path}: {e}")
            else:
                row[f"Quality_{qual}_Score"] = None
                row[f"Quality_{qual}_Label"] = ""
        
        ## Now we can add the row
        rows.append(row)
            
        

# Save the scores to a CSV
if rows:
    df = pd.DataFrame(rows)

    ## No more need to merge with existing labels

    df.to_csv(NEW_RESULT_CSV, index=False)
    print(f"Saved scores to {NEW_RESULT_CSV}.csv")
    display(df.head())
else:
    print("No images were processed. Please check OG_DIRECTORY and VAL_DIRECTORY.")

<All keys matched successfully>
Model for real vs synthetic image detection loaded!


  0%|          | 0/100 [00:00<?, ?it/s]

Processing: 1597
Looking for: JAI_1597_012.png
Found it! Processing...
Looking for: JAI_1597_025.png
Found it! Processing...
Looking for: JAI_1597_050.png
Found it! Processing...
Looking for: JAI_1597_075.png
Found it! Processing...
Looking for: JAI_1597_100.png
Found it! Processing...
Processing: 1971
Looking for: JAI_1971_012.png
Found it! Processing...
Looking for: JAI_1971_025.png
Found it! Processing...
Looking for: JAI_1971_050.png
Found it! Processing...
Looking for: JAI_1971_075.png
Found it! Processing...
Looking for: JAI_1971_100.png
Found it! Processing...
Processing: 3381
Looking for: JAI_3381_012.png
Found it! Processing...
Looking for: JAI_3381_025.png
Found it! Processing...
Looking for: JAI_3381_050.png
Found it! Processing...
Looking for: JAI_3381_075.png
Found it! Processing...
Looking for: JAI_3381_100.png
Found it! Processing...
Processing: 1446
Looking for: JAI_1446_012.png
Found it! Processing...
Looking for: JAI_1446_025.png
Found it! Processing...
Looking for: J

  0%|          | 0/100 [00:00<?, ?it/s]

Processing: 4022
Looking for: JAI_4022_012.png
Found it! Processing...
Looking for: JAI_4022_025.png
Found it! Processing...
Looking for: JAI_4022_050.png
Found it! Processing...
Looking for: JAI_4022_075.png
Found it! Processing...
Looking for: JAI_4022_100.png
Found it! Processing...
Processing: 2166
Looking for: JAI_2166_012.png
Found it! Processing...
Looking for: JAI_2166_025.png
Found it! Processing...
Looking for: JAI_2166_050.png
Found it! Processing...
Looking for: JAI_2166_075.png
Found it! Processing...
Looking for: JAI_2166_100.png
Found it! Processing...
Processing: 6063
Looking for: JAI_6063_012.png
Found it! Processing...
Looking for: JAI_6063_025.png
Found it! Processing...
Looking for: JAI_6063_050.png
Found it! Processing...
Looking for: JAI_6063_075.png
Found it! Processing...
Looking for: JAI_6063_100.png
Found it! Processing...
Processing: 5274
Looking for: JAI_5274_012.png
Found it! Processing...
Looking for: JAI_5274_025.png
Found it! Processing...
Looking for: J

,Img_Name,Actual_Label,Original_Score,Original_Label,Quality_012_Score,Quality_012_Label,Quality_025_Score,Quality_025_Label,Quality_050_Score,Quality_050_Label,Quality_075_Score,Quality_075_Label,Quality_100_Score,Quality_100_Label
0,1597.png,real,-5.105293,real,0.103942,fake,-1.250517,real,-1.239952,real,-1.654388,real,-4.492860,real
1,1971.png,real,-6.876852,real,0.885340,fake,-0.210181,real,-1.149283,real,-1.984002,real,-3.873520,real
2,3381.png,real,-7.421809,real,2.042041,fake,1.355323,fake,-1.119685,real,-1.996179,real,-4.982546,real
3,1446.png,real,-5.134039,real,0.537949,fake,0.210745,fake,0.443108,fake,-0.328182,real,-4.015452,real
4,2590.png,real,-4.540377,real,0.901642,fake,-0.556734,real,-1.100663,real,-1.754714,real,-2.362015,real


Let's evaluate accuracy now

In [7]:
df = pd.read_csv(NEW_RESULT_CSV)
print("--- Accuracy Report ---")

# Evaluate Original Images
if 'Original_Label' in df.columns:
    # Filter out rows where the original failed to process
    valid_df = df[df['Original_Label'].notna() & (df['Original_Label'] != "")]
    correct = (valid_df['Actual_Label'] == valid_df['Original_Label']).sum()
    accuracy = (correct / len(valid_df)) * 100 if len(valid_df) > 0 else 0
    print(f"Original Image Accuracy: {accuracy:.2f}%")
    
# Evaluate Compressed Images
for qual in QUALITIES:
    col_name = f'Quality_{qual}_Label'
    if col_name in df.columns:
        # Filter out rows where this specific compression quality failed/was missing
        valid_df = df[df[col_name].notna() & (df[col_name] != "")]
        correct = (valid_df['Actual_Label'] == valid_df[col_name]).sum()
        accuracy = (correct / len(valid_df)) * 100 if len(valid_df) > 0 else 0
        print(f"Quality {qual} Accuracy:  {accuracy:.2f}%")




--- Accuracy Report ---
Original Image Accuracy: 100.00%
Quality 012 Accuracy:  50.00%
Quality 025 Accuracy:  65.00%
Quality 050 Accuracy:  84.00%
Quality 075 Accuracy:  93.00%
Quality 100 Accuracy:  100.00%


#### WITH FACE TEST

In [10]:
## Instantiate detector. No need to "face only"
detector = RealvsSyntheticDetector(device='cuda:0', M=600, select_face_test=True)

## Extract original images from OG_DIRECTORY
# og_images_dict = extract_images(OG_DIRECTORY)
val_dir_path = Path(VAL_DIRECTORY)
rows = []

REAL_FOLDER_LABEL: str = "real"
FAKE_FOLDER_LABEL: str = "fake"

def get_label(score: float) -> str:
    return FAKE_FOLDER_LABEL if score > 0 else REAL_FOLDER_LABEL

## Now iterate all original images
## To do so, we iterate the folder
for folder_name in [REAL_FOLDER_LABEL, FAKE_FOLDER_LABEL]:
    ## Take the path
    path = Path(NEW_OG_VAL) / folder_name

    ## JPGs are not actually allowed but is added as safety measure
    images = list(path.glob("*.png")) + list(path.glob("*.jpg"))
    
    ## Now iterate all imgs
    for img_path in tqdm(images):
        ## Create the new row
        row = {
            "Img_Name" : img_path.name,
            "Actual_Label" : folder_name
        }

        print(f"Processing: {Path(img_path).stem}")
        ## Now the score
        try:
            img_arr = load_image_as_array(img_path)
            score = float(detector.synth_img_detection(img=img_arr))
            row["Original_Score"] = score
            ## Also save the label directly
            row["Original_Label"] = get_label(score)
        except Exception as e:
            row["Original_Score"] = None
            row["Original_Label"] = ""
            print(f"Error processing original {img_path}: {e}")

        ## At this point, look for the JPEGAI version
        ## Remember to add "rec" to get the actual images
        jpegai_path = Path(NEW_JPEGAI_VAL) / folder_name / "rec"

        for qual in QUALITIES:
            ## First, get the name
            og_name = Path(img_path).stem
            jai_name = f"JAI_{og_name}_{qual}.png"
            print(f"Looking for: {jai_name}")
            jai_path: str = jpegai_path / jai_name
            jai_path: Path = Path(jai_path)

            ## We perfectly know where the img is now
            if jai_path.exists():
                try:
                    print("Found it! Processing...")
                    img_arr = load_image_as_array(str(jai_path))
                    score = float(detector.synth_img_detection(img=img_arr))
                    row[f"Quality_{qual}_Score"] = score
                    ## Also add the label with the same logic
                    row[f"Quality_{qual}_Label"] = get_label(score)
                except Exception as e:
                    row[f"Quality_{qual}_Score"] = None
                    row[f"Quality_{qual}_Label"] = ""
                    print(f"Error processing compressed {jai_path}: {e}")
            else:
                row[f"Quality_{qual}_Score"] = None
                row[f"Quality_{qual}_Label"] = ""
        
        ## Now we can add the row
        rows.append(row)
            
        

# Save the scores to a CSV
if rows:
    df = pd.DataFrame(rows)

    ## No more need to merge with existing labels
    ## Save in a new CSV
    df.to_csv(NEW_FACE_RESULT_CSV, index=False)
    print(f"Saved scores to {NEW_FACE_RESULT_CSV}.csv")
    display(df.head())
else:
    print("No images were processed. Please check OG_DIRECTORY and VAL_DIRECTORY.")

<All keys matched successfully>
Model for real vs synthetic image detection loaded!


  0%|          | 0/182 [00:00<?, ?it/s]

Processing: 5406
Looking for: JAI_5406_012.png
Found it! Processing...
Looking for: JAI_5406_025.png
Found it! Processing...
Looking for: JAI_5406_050.png
Found it! Processing...
Looking for: JAI_5406_075.png
Found it! Processing...
Looking for: JAI_5406_100.png
Found it! Processing...
Processing: 5274
Looking for: JAI_5274_012.png
Found it! Processing...
Looking for: JAI_5274_025.png
Found it! Processing...
Looking for: JAI_5274_050.png
Found it! Processing...
Looking for: JAI_5274_075.png
Found it! Processing...
Looking for: JAI_5274_100.png
Found it! Processing...
Processing: 5421
Looking for: JAI_5421_012.png
Found it! Processing...
Looking for: JAI_5421_025.png
Found it! Processing...
Looking for: JAI_5421_050.png
Found it! Processing...
Looking for: JAI_5421_075.png
Found it! Processing...
Looking for: JAI_5421_100.png
Found it! Processing...
Processing: 5202
Looking for: JAI_5202_012.png
Found it! Processing...
Looking for: JAI_5202_025.png
Found it! Processing...
Looking for: J

,Img_Name,Actual_Label,Original_Score,Original_Label,Quality_012_Score,Quality_012_Label,Quality_025_Score,Quality_025_Label,Quality_050_Score,Quality_050_Label,Quality_075_Score,Quality_075_Label,Quality_100_Score,Quality_100_Label
0,5406.png,real,-2.656124,real,1.185422,fake,0.028996,fake,-0.360525,real,-1.689450,real,-1.151731,real
1,5274.png,real,0.523856,fake,1.854470,fake,0.140991,fake,-0.079091,real,-0.247247,real,0.273781,fake
2,5421.png,real,-1.484032,real,4.208666,fake,1.673185,fake,0.725427,fake,0.577055,fake,-0.375913,real
3,5202.png,real,0.741429,fake,1.997674,fake,-0.118308,real,0.403471,fake,0.391870,fake,1.249809,fake
4,5145.png,real,-0.412828,real,1.081650,fake,-0.956283,real,-0.467754,real,-0.264591,real,-0.062299,real


Now the accuracy test as always

In [7]:
df = pd.read_csv(NEW_FACE_RESULT_CSV)
print("--- Accuracy Report ---")

# Evaluate Original Images
if 'Original_Label' in df.columns:
    # Filter out rows where the original failed to process
    valid_df = df[df['Original_Label'].notna() & (df['Original_Label'] != "")]
    correct = (valid_df['Actual_Label'] == valid_df['Original_Label']).sum()
    accuracy = (correct / len(valid_df)) * 100 if len(valid_df) > 0 else 0
    print(f"Original Image Accuracy: {accuracy:.2f}%")
    
# Evaluate Compressed Images
for qual in QUALITIES:
    col_name = f'Quality_{qual}_Label'
    if col_name in df.columns:
        # Filter out rows where this specific compression quality failed/was missing
        valid_df = df[df[col_name].notna() & (df[col_name] != "")]
        correct = (valid_df['Actual_Label'] == valid_df[col_name]).sum()
        accuracy = (correct / len(valid_df)) * 100 if len(valid_df) > 0 else 0
        print(f"Quality {qual} Accuracy:  {accuracy:.2f}%")

--- Accuracy Report ---
Original Image Accuracy: 67.04%
Quality 012 Accuracy:  35.16%
Quality 025 Accuracy:  57.81%
Quality 050 Accuracy:  53.23%
Quality 075 Accuracy:  53.23%
Quality 100 Accuracy:  56.45%
